In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("analysis").getOrCreate()

In [2]:
df = spark.read.parquet("cleaned_data")
df.show()

+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+----------+----+
|          event_time|      event_type|product_id|        category_id|category_code|        brand|price|  user_id|        user_session| department|    event_timestamp|event_date|hour|
+--------------------+----------------+----------+-------------------+-------------+-------------+-----+---------+--------------------+-----------+-------------------+----------+----+
|2019-10-18 12:08:...|            view|   5730223|1487580007457883075|  accessories|Unknown_brand| 4.44|468808792|cbec98c5-792b-4e9...|accessories|2019-10-18 12:08:00|2019-10-18|  12|
|2019-10-18 12:08:...|            view|   5695491|1487580013581566154|  accessories|        estel|  2.7|169486644|ee44d6fd-7b81-420...|accessories|2019-10-18 12:08:00|2019-10-18|  12|
|2019-10-18 12:08:...|            view|   5809870|1487580009387261981|  accessor

In [3]:
df.describe().show()

+-------+--------------------+----------+------------------+--------------------+-------------------+-------------+------------------+-------------------+--------------------+-----------+-----------------+
|summary|          event_time|event_type|        product_id|         category_id|      category_code|        brand|             price|            user_id|        user_session| department|             hour|
+-------+--------------------+----------+------------------+--------------------+-------------------+-------------+------------------+-------------------+--------------------+-----------+-----------------+
|  count|             4096197|   4096197|           4096197|             4096197|            4096197|      4096197|           4096197|            4096197|             4095560|    4096197|          4096197|
|   mean|                NULL|      NULL| 5467833.478498471|1.545586038596165...|               NULL|         NULL| 8.547736810511498| 5.01357513612451E8|                NULL| 

In [4]:
from pyspark.sql import functions as F

df.select(F.min("event_date")).show()

+---------------+
|min(event_date)|
+---------------+
|     2019-10-01|
+---------------+



In [5]:
# Peak traffic on website
df.groupBy("hour").count().orderBy("count", ascending=False).show(1)

+----+------+
|hour| count|
+----+------+
|  19|264086|
+----+------+
only showing top 1 row


In [6]:
# Let's find which brand product customers added in their cart more
df.filter((df["event_type"] == "cart") & (df["brand"] != "Unknown_brand")).groupBy("brand").count().orderBy("count", ascending=False).show(3)

+------+-----+
| brand|count|
+------+-----+
|runail|98737|
| irisk|78293|
|masura|58657|
+------+-----+
only showing top 3 rows


In [7]:
# Let's find which brand has highest conversion rate of getting purchased
brand_stats = df.groupBy("brand").pivot("event_type").count().fillna(0)

brand_stats = brand_stats.withColumn("conversion_rate", (F.col("purchase") / F.col("view")) * 100)


brand_stats.orderBy("conversion_rate", ascending=False).show(5)

{"ts": "2026-05-07 14:28:18.960", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22012", "context": {"file": "line 4 in cell [8]", "line": "", "fragment": "__truediv__", "errorClass": "DIVIDE_BY_ZERO"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o98.showString.\n: org.apache.spark.SparkArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22012\n== DataFrame ==\n\"__truediv__\" was called from\nline 4 in cell [8]\n\r\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.divideByZeroError(QueryExecutionErrors.scala:205)\r\n\tat org.apache.spark.sql.errors.QueryExecutionErrors.divi

ArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__truediv__" was called from
line 4 in cell [8]
